# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, Duration, Channel]  

 2. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 3. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

##  Module Installation

In [ ]:
%%capture
!pip install yt-dlp youtube-transcript-api pydub
!apt-get update
!apt-get install -y ffmpeg

## Module Imports

In [ ]:
import yt_dlp
import pandas as pd
import youtube_transcript_api

### Example Usage: Get Channel Link from Title

Enter the channel title below to find its YouTube link. You can modify the `channel_title_to_find` variable.

In [ ]:
# channel_title_to_find = "T-Series"  # Replace with the actual channel title you want to search for

# found_channel_link = get_channel_link_from_title(channel_title_to_find)

# if found_channel_link:
#     print(f"\nThe YouTube link for '{channel_title_to_find}' is: {found_channel_link}")
# else:
#     print(f"\nNo YouTube link found for '{channel_title_to_find}'. Please check the title.")

## Main Variables

In [ ]:
video_channel = []
video_IDs = []
video_durations = []
video_language = []
videos_LARGE = []
video_titles = []
video_language_INFO = {
    'Hindi'                                     : "hi-IN",
    'Kashmiri'                                  : "ks-IN",
    'Assamese'                                  : "as-IN",
    "Marathi"                                   : "mr-IN",
    "Gujarati"                                  : "gu-IN",
    "Kannada"                                   : "kn-IN",
    "Odia"                                      :"od-IN",
    "Bengali"                                   :"bn-IN"
}

### Constant for yt_dlp

In [ ]:
ydl_opts = {
                'extract_flat': True,
                'skip_download': True,
                'ignoreerrors': True,
                # 'http_headers': {
                #     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                #     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                #     'Accept-Language': 'en-us,en;q=0.5',
                #     'Sec-Fetch-Mode': 'navigate'
                # },
                # 'geo_bypass_country': 'US' # Or any other country code
            }

In [ ]:
#Helper Functions

def Larger_Video_CHECK(VIDEO_ID: str, DURATION: int):
    """Helper function to move larger videos to another area for processing later on"""
    if(DURATION > 1800):
        videos_LARGE.append(VIDEO_ID)
        return True

    return False


def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID



## Get YT Channel Usernames and Playlists

In [ ]:
Sources = []

print("Please Provide the complete clean links")
def Get_Sources():
    while(1):
        print("Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):")
        a = input().strip()
        if(a in ['N','n']):
            break;

        if a=='': continue

        Sources.append(a)

Get_Sources()

Please Provide the complete clean links
Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):
https://www.youtube.com/playlist?list=PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb
Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):
n


### Function to fetch and extract video details to formulate the cvs dataset

In [ ]:
### Start Video ID, duration, extraction

def extract_video_details(Video_URL : str):

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(Video_URL, download=False)

        for entry in info['entries']:
            video_channel.append(info['channel'])
            video_IDs.append(entry['id'])
            video_durations.append(entry['duration'])
            video_titles.append(entry['title'])

In [ ]:
for source in Sources:
    extract_video_details(source)

print(len(video_channel) == len(video_durations) ==len(video_IDs))

[youtube:tab] Extracting URL: https://www.youtube.com/playlist?list=PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb
[youtube:tab] PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb: Downloading webpage
[youtube:tab] PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: Krishi Darshan
[youtube:tab] PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb page 1: Downloading API JSON
[youtube:tab] Playlist Krishi Darshan: Downloading 74 items of 74
[download] Downloading item 1 of 74
[download] Downloading item 2 of 74
[download] Downloading item 3 of 74
[download] Downloading item 4 of 74
[download] Downloading item 5 of 74
[download] Downloading item 6 of 74
[download] Downloading item 7 of 74
[download] Downloading item 8 of 74
[download] Downloading item 9 of 74
[download] Downloading item 10 of 74
[download] Downloading item 11 of 74
[download] Downloading item 12 of 74
[download] Downloading item 13 of 74
[download] Downloading item 14 of 74
[download] Downloadi

### Form the .csv Dataset

In [ ]:
##Form csv dataset
#.csv dataset creation with [SrNo. , YT_VIDEO_ID, Duration, CHANNEL]
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "YT_VIDEO_ID": video_IDs,
    "YT_VIDEO_TITLE": video_titles,
    "YT_VIDEO_LINK": [YT_Link_Generator(ID) for ID in video_IDs],
    "Duration": video_durations,
    "Channel": video_channel
})

df.to_csv("YT_DATASET.csv", index = False)

### DOWNLOAD AUDIO

In [ ]:
# %%capture #Comment out if you dont want any outputs
def Don_Eladio(video_ID, video_Duration):
    flag = Larger_Video_CHECK(video_ID, video_Duration)

    if(flag): return

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': f'KrishiDarshan/{video_ID}/audio.%(ext)s',
        # 'http_headers': {
        #     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        #     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        #     'Accept-Language': 'en-us,en;q=0.5',
        #     'Sec-Fetch-Mode': 'navigate'
        # },
        # 'geo_bypass_country': 'US' # Or any other country code
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])


l = len(video_IDs)
for i in range(l):
    Don_Eladio(video_IDs[i], video_durations[i])

[youtube] Extracting URL: https://www.youtube.com/watch?v=SxgaGXcQ8ZY
[youtube] SxgaGXcQ8ZY: Downloading webpage


[youtube] SxgaGXcQ8ZY: Downloading android vr player API JSON
[info] SxgaGXcQ8ZY: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/SxgaGXcQ8ZY/audio.m4a
[download] 100% of   20.85MiB in 00:00:01 at 11.91MiB/s  
[FixupM4a] Correcting container of "KrishiDarshan/SxgaGXcQ8ZY/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/SxgaGXcQ8ZY/audio.mp3
Deleting original file KrishiDarshan/SxgaGXcQ8ZY/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=JLwhLr4ZVd0
[youtube] JLwhLr4ZVd0: Downloading webpage


[youtube] JLwhLr4ZVd0: Downloading android vr player API JSON
[info] JLwhLr4ZVd0: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/JLwhLr4ZVd0/audio.m4a
[download] 100% of   22.29MiB in 00:00:01 at 12.76MiB/s  
[FixupM4a] Correcting container of "KrishiDarshan/JLwhLr4ZVd0/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/JLwhLr4ZVd0/audio.mp3
Deleting original file KrishiDarshan/JLwhLr4ZVd0/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=ET8rwerJTg0
[youtube] ET8rwerJTg0: Downloading webpage


[youtube] ET8rwerJTg0: Downloading android vr player API JSON
[info] ET8rwerJTg0: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/ET8rwerJTg0/audio.m4a
[download] 100% of   23.01MiB in 00:00:02 at 9.15MiB/s   
[FixupM4a] Correcting container of "KrishiDarshan/ET8rwerJTg0/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/ET8rwerJTg0/audio.mp3
Deleting original file KrishiDarshan/ET8rwerJTg0/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=BfMmAW-58ng
[youtube] BfMmAW-58ng: Downloading webpage


[youtube] BfMmAW-58ng: Downloading android vr player API JSON
[info] BfMmAW-58ng: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/BfMmAW-58ng/audio.m4a
[download] 100% of   18.83MiB in 00:00:02 at 6.58MiB/s   
[FixupM4a] Correcting container of "KrishiDarshan/BfMmAW-58ng/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/BfMmAW-58ng/audio.mp3
Deleting original file KrishiDarshan/BfMmAW-58ng/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=VzSq2eW8ZG8
[youtube] VzSq2eW8ZG8: Downloading webpage


...

[youtube] RkkFAMVk8bE: Downloading android vr player API JSON
[info] RkkFAMVk8bE: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/RkkFAMVk8bE/audio.m4a
[download] 100% of    8.53MiB in 00:00:02 at 4.01MiB/s   
[FixupM4a] Correcting container of "KrishiDarshan/RkkFAMVk8bE/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/RkkFAMVk8bE/audio.mp3
Deleting original file KrishiDarshan/RkkFAMVk8bE/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=Uql55tbZ4Ss
[youtube] Uql55tbZ4Ss: Downloading webpage


[youtube] Uql55tbZ4Ss: Downloading android vr player API JSON
[info] Uql55tbZ4Ss: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/Uql55tbZ4Ss/audio.m4a
[download] 100% of   26.59MiB in 00:00:02 at 12.17MiB/s  
[FixupM4a] Correcting container of "KrishiDarshan/Uql55tbZ4Ss/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/Uql55tbZ4Ss/audio.mp3
Deleting original file KrishiDarshan/Uql55tbZ4Ss/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=b22kbPH7MA4
[youtube] b22kbPH7MA4: Downloading webpage


[youtube] b22kbPH7MA4: Downloading android vr player API JSON
[info] b22kbPH7MA4: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/b22kbPH7MA4/audio.m4a
[download] 100% of   25.04MiB in 00:00:01 at 18.43MiB/s  
[FixupM4a] Correcting container of "KrishiDarshan/b22kbPH7MA4/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/b22kbPH7MA4/audio.mp3
Deleting original file KrishiDarshan/b22kbPH7MA4/audio.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=ftJqIlS3M1g
[youtube] ftJqIlS3M1g: Downloading webpage


[youtube] ftJqIlS3M1g: Downloading android vr player API JSON
[info] ftJqIlS3M1g: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/ftJqIlS3M1g/audio.m4a
[download] 100% of   25.64MiB in 00:00:01 at 13.44MiB/s  
[FixupM4a] Correcting container of "KrishiDarshan/ftJqIlS3M1g/audio.m4a"
[ExtractAudio] Destination: KrishiDarshan/ftJqIlS3M1g/audio.mp3


KeyboardInterrupt: 

In [ ]:
# %%capture
# for i in range(len(video_IDs[:1])):
#     flag = Larger_Video_CHECK(video_IDs[i], video_durations[i])

#     if(flag): continue

#     ydl_opts = {
#         'format': 'bestaudio/best',
#         'postprocessors': [{
#             'key': 'FFmpegExtractAudio',
#             'preferredcodec': 'mp3',
#             'preferredquality': '192',
#         }],
#         'outtmpl': f'KrishiDarshan/{video_IDs[i]}/audio.%(ext)s',
#     }

#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         ydl.download([YT_Link_Generator(video_IDs[i])])

### Start Generating the transcript using whisper large v3 from Hugging Face

> Implement a Batch Processing Architecture to process the Audio files for highest possible accuracy in the transcripts.

In [ ]:
# WHisper works best with 30 second splits of the audio files fed to the api batch after batch

### The splitter will work to split all the splits first and then start transcription successively.
### Transcripts of each split will be added into the

In [ ]:
from pydub import AudioSegment

def splitter(video_Duration):
    # there is an edge case here which will only occur when an audio file is smaller than 30 seconds
    if (video_Duration < 30):
        return


    """Take the audio file duration, and return a list of timestamps with seperation of 30"""
    return [i+(video_Duration - i) if (video_Duration - i) <= 29 else i  for i in range(0,video_Duration+1, 30)]

# print(splitter(59))

def audio_splitter(audio_file, start, end): # start and end must be in milliseconds
    audio = AudioSegment.from_file(audio_file)
    print(len(audio))

    segment = audio[start:end]
    segment.export(f"/content/splits/segment{start}_{end}.mp3", format='mp3')

In [ ]:
# Fetch the audio files and start transcription


174721


In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 7.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

result = pipe(segment)
print(result["text"])


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


NameError: name 'segment' is not defined

In [ ]:
result = pipe('/content/segment.mp3', language='hi')
print(result["text"])

 प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्रस्तुति प्र
